<a href="https://colab.research.google.com/github/George-Mironenko/credit_model/blob/main/credit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install matplotlib

In [ ]:
!pip install pandas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("/content/german_credit_data.csv", index_col=False)
df.head(10)

,Unnamed: 0,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,0,67,male,2,own,NaN,little,1169,6,radio/TV,good
1,1,22,female,2,own,little,moderate,5951,48,radio/TV,bad
2,2,49,male,1,own,little,NaN,2096,12,education,good
3,3,45,male,2,free,little,little,7882,42,furniture/equipment,good
4,4,53,male,2,free,little,little,4870,24,car,bad
5,5,35,male,1,free,NaN,NaN,9055,36,education,good
6,6,53,male,2,own,quite rich,NaN,2835,24,furniture/equipment,good
7,7,35,male,3,rent,little,moderate,6948,36,car,good
8,8,61,male,1,own,rich,NaN,3059,12,radio/TV,good
9,9,28,male,3,own,little,moderate,5234,30,car,bad


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

Mapp

In [ ]:
mapping = {
    'own':3,
    'rent': 2,
    'free':1
}
df['Housing_int'] = df['Housing'].map(mapping)

In [ ]:
mapping = {
    'good':1,
    'bad':0
}
df['Risk_int'] = df['Risk'].map(mapping)

In [ ]:
X = df[['Age', 'Duration', 'Credit amount', 'Job', 'Housing_int']]

y = df['Risk_int']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.2, random_state=42
)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=50, # Сколько всего деревьев
    max_depth=5,   # Максимальная глубина (сколько if максимум сделать)
    n_jobs=-1, # Используем все процесоры
    random_state=42,
    oob_score=True,
    min_samples_leaf=3 # минимальное обьектов - тоесть столбцов в условии if
)

In [ ]:
rf.fit(X_train, y_train)

RandomForestClassifier(max_depth=5, min_samples_leaf=3, n_estimators=50,
                       n_jobs=-1, oob_score=True, random_state=42)

In [ ]:
print(f"OOB Score: {rf.oob_score_:.3f}")

OOB Score: 0.713


In [ ]:
from sklearn.metrics import accuracy_score

print(f"OOB Score: {rf.oob_score_:.3f}")
print(f"Train Acc: {accuracy_score(y_train, rf.predict(X_train)):.3f}")
print(f"Test Acc:  {accuracy_score(y_test, rf.predict(X_test)):.3f}")

OOB Score: 0.713
Train Acc: 0.765
Test Acc:  0.715


И так. Модель впринцеме номально все предсказала

In [ ]:
from sklearn.model_selection import GridSearchCV

# Сетка параметров для твоих 1000 строк
param_grid = {
    'n_estimators': [50, 100, 200],      # Количество деревьев
    'max_depth': [5, 7, 10, None],       # Глубина (None = без ограничений)
    'min_samples_split': [2, 5, 10],     # Минимум для разделения узла
    'min_samples_leaf': [1, 3, 5],       # Минимум в листе
    'max_features': ['sqrt', 'log2', 0.3, 0.5]  # Сколько признаков брать
}

rf_base = RandomForestClassifier(random_state=42, oob_score=True, n_jobs=-1)

grid_search = GridSearchCV(
    rf_base,
    param_grid,
    cv=5,                    # 5-кратная кросс-валидация
    scoring='accuracy',      # или 'roc_auc' для бинарной
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Лучшие параметры: {grid_search.best_params_}")
print(f"Лучшее качество: {grid_search.best_score_:.3f}")

# Используй лучшую модель
rf_best = grid_search.best_estimator_

  И мы решили сделать много моделей что бы найти закономерности.

  - Что можно улучшить первое это cross validation - 5 это середина. Но если мы постави 3 тогда будет больше моделей которые обучаються на одиноковых данных. У каждой третьей будет одинакове данные. Так сибе. Но это быстро. А чем нам грозит то что данные будут одиноковыми. Тем что теперь модели как мне кажеться будут почти одинаковы, и не будут отличаться почти друг от друга. А если мы cv=10 то тогда унас разнообразие данные будет сильнее но это будет дольше и дороже в обучении.

  - Также выжно смотреть на *scoring* то что мы протестировали и нашли это не очем не говирит о том как модель хорошо предсказывает класс good, или если она сказал bad каков шанс что это так. Что нам надо ещё понять какой параметр нам важнее.

In [ ]:
rf_good = RandomForestClassifier(
    n_estimators=50, # Сколько всего деревьев
    max_depth=7,   # Максимальная глубина (сколько if максимум сделать)
    n_jobs=-1, # Используем все процесоры
    random_state=42,
    max_features='sqrt',
    oob_score=True,
    min_samples_split=5,
    min_samples_leaf=2 # минимальное обьектов - тоесть столбцов в условии if
)

In [ ]:
rf_good.fit(X_train, y_train)

RandomForestClassifier(max_depth=7, min_samples_leaf=2, min_samples_split=5,
                       n_estimators=50, n_jobs=-1, oob_score=True,
                       random_state=42)

In [ ]:

print(f"OOB Score: {rf_good.oob_score_:.3f}")
print(f"Train Acc: {accuracy_score(y_train, rf_good.predict(X_train)):.3f}")
print(f"Test Acc:  {accuracy_score(y_test, rf_good.predict(X_test)):.3f}")

OOB Score: 0.711
Train Acc: 0.815
Test Acc:  0.700


In [ ]:
rf_regularized = RandomForestClassifier(
    n_estimators=100,         # увеличил для стабильности
    max_depth=4,              # уменьшил глубину (еще проще)
    min_samples_leaf=5,       # увеличил (еще грубее)
    min_samples_split=10,     # добавил
    max_features=0.5,         # берем только 50% признаков
    oob_score=True,
    random_state=42
)

In [ ]:
rf_regularized.fit(X_train, y_train)

RandomForestClassifier(max_depth=4, max_features=0.5, min_samples_leaf=5,
                       min_samples_split=10, oob_score=True, random_state=42)

In [ ]:
print(f"OOB Score: {rf_good.oob_score_:.3f}")
print(f"Train Acc: {accuracy_score(y_train, rf_good.predict(X_train)):.3f}")
print(f"Test Acc:  {accuracy_score(y_test, rf_good.predict(X_test)):.3f}")

OOB Score: 0.711
Train Acc: 0.815
Test Acc:  0.700


In [ ]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 15.7 MB/s eta 0:00:00


In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

# Настройка (создаст папку mlruns/)
mlflow.set_experiment("credit_risk_rf")

with mlflow.start_run(run_name="rf_50_7_depth"):

    # === 1. Логируем параметры ===
    mlflow.log_params({
        "n_estimators": 50,
        "max_depth": 7,
        "max_features": "sqrt",
        "min_samples_split": 5,
        "min_samples_leaf": 2,
        "oob_score": True,
        "random_state": 42
    })

    # === 2. Обучаем модель ===
    rf_good = RandomForestClassifier(
        n_estimators=50,
        max_depth=7,
        n_jobs=-1,
        random_state=42,
        max_features='sqrt',
        oob_score=True,
        min_samples_split=5,
        min_samples_leaf=2
    )
    rf_good.fit(X_train, y_train)

    # === 3. Логируем OOB score ===
    mlflow.log_metric("oob_score", rf_good.oob_score_)

    # === 4. Предсказания и метрики ===
    y_pred = rf_good.predict(X_test)
    y_proba = rf_good.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, pos_label='good'),
        "recall": recall_score(y_test, y_pred, pos_label='good'),
        "roc_auc": roc_auc_score(y_test, y_proba)
    }

    for name, value in metrics.items():
        mlflow.log_metric(name, value)

    # === 5. Логируем саму модель ===
    mlflow.sklearn.log_model(
        sk_model=rf_good,
        artifact_path="random_forest_model",
        input_example=X_test.iloc[:5]  # пример входных данных
    )

    # === 6. Логируем важность признаков (как артефакт) ===
    import matplotlib.pyplot as plt

    feature_importance = pd.DataFrame({
        'feature': feature_names,  # твой список названий колонок
        'importance': rf_good.feature_importances_
    }).sort_values('importance', ascending=False)

    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance['feature'][:15], feature_importance['importance'][:15])
    plt.xlabel('Importance')
    plt.title('Top 15 Feature Importances')
    plt.tight_layout()

    mlflow.log_figure(plt.gcf(), "feature_importance.png")

    print(f"Run ID: {mlflow.active_run().info.run_id}")
    print(f"OOB Score: {rf_good.oob_score_:.3f}")
    print(f"Test AUC: {metrics['roc_auc']:.3f}")

ValueError: pos_label=good is not a valid label. It should be one of [0, 1]

In [ ]:
y_train_encoded = y_train.map({'bad': 0, 'good': 1})
y_test_encoded = y_test.map({'bad': 0, 'good': 1})

In [ ]:
!mlflow.db

/bin/bash: line 1: mlflow.db: command not found


In [ ]:
!mlflow ui

Backend store URI not provided. Using sqlite:///mlflow.db
Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
INFO:     Started parent process [6059]
2026/04/03 11:55:42 INFO mlflow.server.jobs.utils: Starting huey consumer for job function run_online_session_scorer
2026/04/03 11:55:42 INFO mlflow.server.jobs.utils: Starting huey consumer for job function optimize_prompts
2026/04/03 11:55:42 INFO mlflow.server.jobs.utils: Starting huey consumer for job function run_online_trace_scorer
2026/04/03 11:55:42 INFO mlflow.server.jobs.utils: Starting huey consumer for job function invoke_scorer
2026/04/03 11:55:42 INFO mlflow.server.jobs.utils: Starting dedicated Huey consumer for periodic tasks
INFO:     Started server proc